# Link2iOS — Breath of the Wild (BOTW)

Notebook organizado no mesmo fluxo por etapas do Kleo2iOS, mas com o pipeline NX2Apple Route C deste projeto.

**Dois caminhos:**
- **Retomar:** autenticar o Drive, clonar os scripts e executar a célula de recuperação. Ela encontra apenas manifests `Link2iOS_BOTW_*` e recupera a preparação BOTW já salva, sem baixar o pacote base/NSPs nem repetir extração, rewrite ou shaders.
- **Primeira preparação:** executar a etapa longa uma vez. Ela valida os 4 pacotes (1 NSP base + update + 2 DLCs), gera RomFS/AOC, salva o workspace e publica um `handoff.json` de recuperação no Drive.

A pasta de origem também contém dados do Kleo2iOS. O script usa somente `prod.keys`, `update.nsp`, `dlc1.nsp`, `dlc2.nsp` e a pasta `BOTW_Base_NSP` (aceita também `BOTW_Base_XCI` de execuções anteriores); todos os outros itens são ignorados. O código não reutiliza arquivos de trabalho de outros jogos.

**Velocidade/transferência:** não há percentual estimado artificial. Cada transferência registra bytes reais, tamanho validado e tempo decorrido. Um clone do repositório ou a autenticação do Drive não contam como download do jogo.

In [ ]:
import os, sys, argparse, json
from pathlib import Path
from getpass import getpass
from google.colab import auth
import google.auth
from googleapiclient.discovery import build

DRIVE_FOLDER_ID = "19HXEJCK-26NfBVKeUP3wqFxMFpoF3ykm"
ROOT = Path("/content/link2ios")

auth.authenticate_user()
DRIVE_CREDENTIALS, _ = google.auth.default(
    scopes=["https://www.googleapis.com/auth/drive"]
)
drive_api_client = build(
    "drive", "v3", credentials=DRIVE_CREDENTIALS, cache_discovery=False
)
folder = drive_api_client.files().get(
    fileId=DRIVE_FOLDER_ID, fields="id,name,mimeType"
).execute()
assert folder.get("mimeType") == "application/vnd.google-apps.folder", folder
print("Google Drive autenticado:", folder.get("name"))
print("Autenticação concluída; nenhum jogo foi baixado nesta célula.")

## 1. Obter o pipeline Link2iOS

Clona somente a branch `Link2iOS`. O checkout pequeno de scripts não é o download dos arquivos do jogo.

In [ ]:
%cd /content
!rm -rf /content/Link2iOS-repo
!git clone --depth 1 --branch Link2iOS https://github.com/joao9vegeto-stack/KintTany.git /content/Link2iOS-repo
!git -C /content/Link2iOS-repo rev-parse --short HEAD

## 2. Recuperar uma preparação BOTW já concluída (sem refazer)

Execute esta célula primeiro quando estiver retomando uma sessão interrompida ou quando a preparação já tiver sido concluída antes. Ela procura exclusivamente uma pasta `Link2iOS_BOTW_*` na pasta compartilhada, lê o pequeno `handoff.json` e recupera o estado local.

**Não baixa os arquivos do jogo, não extrai NSP/XCI, não executa AOT rewrite nem shader split.** Pastas `Kleo2iOS_GameData_*` e arquivos `Kleo2iOS-*` são ignorados.

In [ ]:
%cd /content/Link2iOS-repo
import importlib, hashlib, subprocess
sys.path.insert(0, "/content/Link2iOS-repo/colab")
# Colab kernels keep imported modules in sys.modules even after the repository is recloned.
# Remove cached modules so this run cannot execute source_archive.py from an older checkout.
for module_name in ("source_archive", "prepare_linux", "run_colab"):
    sys.modules.pop(module_name, None)
importlib.invalidate_caches()
repo_head = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd="/content/Link2iOS-repo", text=True
).strip()
import source_archive
import run_colab
expected_source = Path("/content/Link2iOS-repo/colab/source_archive.py").resolve()
loaded_source = Path(source_archive.__file__).resolve()
assert loaded_source == expected_source, f"source_archive carregado de caminho inesperado: {loaded_source}"
source_text = loaded_source.read_text(encoding="utf-8")
assert "XCI dividido detectado" in source_text and "validate_nsp" in source_text and "BOTW_Base_NSP" in source_text, (
    "O checkout do Colab não contém o suporte ao NSP base e o diagnóstico de XCI dividido. "
    f"HEAD={repo_head}; arquivo={loaded_source}"
)
print("[Link2iOS] Commit do código:", repo_head)
print("[Link2iOS] Módulo de origem:", loaded_source)
print("[Link2iOS] SHA-256 source_archive.py:", hashlib.sha256(loaded_source.read_bytes()).hexdigest())
RECOVERED_HANDOFF = run_colab.recover(
    argparse.Namespace(drive_folder_id=DRIVE_FOLDER_ID),
    api=drive_api_client,
)
HANDOFF_READY = RECOVERED_HANDOFF is not None


## 3. Primeira preparação Linux (executar somente se não houver recuperação)

Esta é a etapa demorada. O Colab obtém o NSP base da pasta `BOTW_Base_NSP` no Drive (ou baixa o 7z uma vez e extrai o único NSP base). Também aceita XCI base em execuções anteriores. Lê ainda `update.nsp`, `dlc1.nsp`, `dlc2.nsp` e `prod.keys`.

O NXNative executa a leitura dos metadados, extração do jogo, rewrite AArch64 e separação dos shaders NVN. Depois, o Colab envia RomFS, AOC e workspace ao Drive e grava o manifest de recuperação. O progresso mostra bytes reais e os tempos de transferência.

**Não execute esta célula se a etapa 2 recuperou uma preparação válida.**

In [ ]:
if HANDOFF_READY:
    print("Preparação BOTW válida recuperada. Pulando download, extração, rewrite e shader split.")
else:
    download_url = ""
    base_folder_exists = any(
        item.get("mimeType") == "application/vnd.google-apps.folder"
        and item.get("name", "").lower().startswith(("botw_base_nsp", "botw_base_xci"))
        for item in run_colab.list_children(drive_api_client, DRIVE_FOLDER_ID)
    )
    cached_archive = (ROOT / "7z-temporary" / "botw-base.7z").is_file()
    if not base_folder_exists and not cached_archive:
        download_url = getpass("Cole o link assinado do 7z com o NSP base: ").strip()
        assert download_url.startswith(("https://", "http://")), "URL 7z válida obrigatória na primeira preparação."
        os.environ["LINK2IOS_DOWNLOAD_URL"] = download_url
    else:
        os.environ.pop("LINK2IOS_DOWNLOAD_URL", None)
        if cached_archive and not base_folder_exists:
            print("7z já existe nesta sessão do Colab; será reutilizado sem novo download.")
        else:
            print("Pacote base já está no Drive; não haverá novo download do 7z.")
    del download_url
    run_colab.prepare(
        argparse.Namespace(drive_folder_id=DRIVE_FOLDER_ID),
        api=drive_api_client,
    )
    HANDOFF_PATH = Path("/content/link2ios-handoff.json")
    assert HANDOFF_PATH.is_file(), "A preparação terminou, mas o arquivo local de handoff não foi criado."
    RECOVERED_HANDOFF = json.loads(HANDOFF_PATH.read_text(encoding="utf-8"))
    HANDOFF_READY = (
        RECOVERED_HANDOFF.get("project") == "Link2iOS"
        and RECOVERED_HANDOFF.get("title_id") == "01007EF00011E000"
        and RECOVERED_HANDOFF.get("package_version") == "v1114112"
    )
    assert HANDOFF_READY, "O handoff local não corresponde ao BOTW 1.9.0."

## 4. Verificação do resultado e checkpoint de recuperação

O manifest persistido confirma Title ID, versão, archive do workspace, RomFS e quantidade de imagens AOC. Se o runtime do Colab reiniciar depois, basta repetir autenticação + clone + recuperação; não execute de novo a etapa longa.

In [ ]:
HANDOFF_PATH = Path("/content/link2ios-handoff.json")
assert HANDOFF_READY and HANDOFF_PATH.is_file(), "Nenhum handoff BOTW recuperado/gerado."
HANDOFF = json.loads(HANDOFF_PATH.read_text(encoding="utf-8"))
assert HANDOFF["title_id"] == "01007EF00011E000"
assert HANDOFF["package_version"] == "v1114112"
assert HANDOFF["branch"] == "Link2iOS"
expected_aoc = {"1.img", "2.img"}
assert {x["name"] for x in HANDOFF["game_data"]["aoc"]} == expected_aoc, "Drive handoff precisa conter aoc/1.img e aoc/2.img."
assert {x["name"] for x in HANDOFF["prepared"]["aoc_images"]} == expected_aoc, "Colab workspace precisa registrar as duas imagens AOC."
assert HANDOFF["prepared"]["aoc_title_ids"] == {
    "1.img": "01007EF00011F001",
    "2.img": "01007EF00011F002",
}
print("BOTW 1.9.0 — preparação verificada.")
print("Request:", HANDOFF["request_id"])
print("Drive:", HANDOFF.get("drive_output_folder_url"))
print("Workspace:", f'{HANDOFF["work_archive"]["size"]:,} bytes')
print("RomFS:", f'{HANDOFF["game_data"]["romfs"]["size"]:,} bytes')
print("AOC images:", len(HANDOFF["game_data"]["aoc"]))
print("Pronto para o handoff do GitHub Actions.")

## 5. Última célula: enviar handoff e disparar somente Link2iOS

Use o Fine-grained PAT com `Contents: Read and write` restrito ao repositório. Esta célula envia apenas `handoff.json` para a branch `Link2iOS`; o workflow final baixa e valida os artefatos preparados no Drive. Não altera `main` nem `Kleo2iOS`.

In [ ]:
%cd /content/Link2iOS-repo
!python colab/run_colab.py dispatch --handoff /content/link2ios-handoff.json